In [12]:
%load_ext autoreload
%autoreload 2

In [24]:
from graphdatascience import GraphDataScience
from neo4j import GraphDatabase
from utils.database_utils import (
    generate_database_and_retriever,
    populate_community_database,
    populate_node_db,
)

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")

In [14]:
gds = GraphDataScience(URI, AUTH)

In [15]:
all_labels = gds.run_cypher("CALL db.labels()")["label"].tolist()
all_labels = [
    lab for lab in all_labels if lab not in ["image", "text", "table", "file"]
]
all_rels = gds.run_cypher("CALL db.relationshipTypes()")["relationshipType"].tolist()
rel_config = {rel: {"orientation": "UNDIRECTED"} for rel in all_rels}

In [16]:
G, result = gds.graph.project(
    "my-community-graph",
    all_labels,  # From the previous step
    rel_config,  # Every relationship type in the DB
)

In [17]:
global_communities = gds.leiden.write(
    G,
    writeProperty="globalCommunityId",
    includeIntermediateCommunities=False,
    relationshipWeightProperty=None,
    gamma=0.2,
)


In [18]:
mid_communities = gds.leiden.write(
    G,
    writeProperty="midCommunityId",
    includeIntermediateCommunities=False,
    relationshipWeightProperty=None,
    gamma=1.5,
)

In [19]:
gds.graph.drop(G)

graphName                                               my-community-graph
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                               60
relationshipCount                                                      122
configuration            {'relationshipProjection': {'IS_AFFILIATED_WIT...
density                                                           0.034463
creationTime                           2026-03-18T06:26:07.695932794+00:00
modificationTime                       2026-03-18T06:26:07.695932794+00:00
schema                   {'graphProperties': {}, 'relationships': {'IS_...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'IS_...
Name: 0, dtype: object

# Fetch all the community nodes and generate a summary

In [21]:
def fetch_community_data(driver, level="global"):

    assert level in ["global", "mid"], "level must be either 'global' or 'mid'"
    level_query = """
    WHERE n.{level}CommunityId IS NOT NULL
    WITH n.{level}CommunityId AS communityId, collect(n) AS nodes
    """
    level_query = level_query.format(level=level)
    query = (
        """
    MATCH (n)
    """
        + level_query
        + """
    UNWIND nodes AS source
    MATCH (source)-[r]->(target)
    WHERE target IN nodes
    RETURN communityId, 
           [node IN nodes | node.name] AS entity_names, 
           collect({s: source.name, t: type(r), o: target.name}) AS triples
    """
    )
    with driver.session() as session:
        records = session.run(query)
        community_inputs = {}

        for record in records:
            # Format the entities and relationships into a text block
            entities_str = ", ".join(record["entity_names"])
            triples_str = "\n".join(
                [f"- {t['s']} --[{t['t']}]--> {t['o']}" for t in record["triples"]]
            )

            community_report_input = {
                "community_id": record["communityId"],
                "nodes": entities_str,
                "relationships": triples_str,
            }
            community_inputs[record["communityId"]] = community_report_input

        return community_inputs


def fetch_nodes_data(driver):
    query = """
    MATCH (n)
    WHERE NOT any(label IN labels(n) WHERE label IN ["file", "text", "image", "table"])
    RETURN 
        n.name AS name, 
        labels(n)[0] AS type, 
        n.description AS description
    """

    with driver.session() as session:
        result = session.run(query)
        # We iterate through the records and create a list of dictionaries
        nodes = [
            {
                "name": record["name"],
                "type": record["type"],
                "description": record["description"],
            }
            for record in result
        ]

    return nodes

In [9]:
from neo4j import Driver

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    global_community_inputs = fetch_community_data(driver, level="global")
    mid_community_inputs = fetch_community_data(driver, level="mid")


In [10]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser
from dataclasses import dataclass, field


@dataclass
class KGCommunity:
    id: int
    title: str
    summary: str
    full_report: str
    nodes: str
    relationships: str
    metadata: dict = field(default_factory=dict)


SYSTEM_INSTRUCTION = """
You are an expert Graph Data Analyst. Your task is to synthesize information about a specific community of entities into a structured report.

### Instructions
1. **Title**: Create a concise, descriptive title that captures the community's essence.
2. **Summary**: Provide a high-level overview (1-2 sentences) of the core theme and why these entities are grouped together.
4. **Relationship Dynamics**: Describe the primary interactions. How do these entities support, compete with, or interact with one another?

### Output Format
Return your response in clean Markdown. Use headers for sections and bold text for entity names. Do not include any conversational filler.
"""
PROMPT = """
### Community Data
**Nodes within this community:**
{nodes}

**Relationships/Edges:**
{relationships}

Please generate the community report based on the System Instructions.
"""


class CommunitySummarizer:
    def __init__(self, model_name="gemma3:12b"):
        self.model_name = model_name
        self.chain = OllamaLLM(model=model_name)

    def find_comunity_summary(self, community_inputs):
        for community_id, community_data in community_inputs.items():
            # Safely extract data, defaulting to empty strings if missing
            nodes = community_data.get("nodes", "No node data available.")
            relationships = community_data.get(
                "relationships", "No relationship data available."
            )

            # Format the specific prompt for this iteration
            formatted_prompt = PROMPT.format(nodes=nodes, relationships=relationships)

            # Build the message payload
            messages = [
                SystemMessage(content=SYSTEM_INSTRUCTION),
                HumanMessage(content=formatted_prompt),
            ]

            # Generate and store the result
            print(f"Generating summary for community {community_id}")
            summary = self.chain.invoke(messages)
            community_inputs[community_id]["summary"] = summary


def transform_dicts_to_objects(community_dict):
    community_objects = []

    for comm_id, data in community_dict.items():
        # Extract a title from the Markdown summary (e.g., "### Analysis...")
        # If no header exists, fallback to a generic title
        summary_lines = data.get("summary", "").split("\n")
        title = (
            summary_lines[0].replace("#", "").strip()
            if summary_lines
            else f"Community {comm_id}"
        )

        # Build the 'full_report' by combining nodes, relationships, and the summary
        # This ensures the LLM gets everything it needs from the docstore
        full_report = (
            f"COMMUNITY NODES: {data.get('nodes')}\n\n"
            f"RELATIONSHIPS:\n{data.get('relationships')}\n\n"
            f"{data.get('summary')}"
        )

        obj = KGCommunity(
            id=comm_id,
            title=title,
            summary=data.get("summary", ""),
            full_report=full_report,
            nodes=data.get("nodes", ""),
            relationships=data.get("relationships", ""),
            metadata={"source": "kg_extraction", "community_id": comm_id},
        )
        community_objects.append(obj)

    return community_objects


In [11]:
communtiy_summarizer = CommunitySummarizer(model_name="gemma3:12b")

# Create a database for the communities: 
The goal is to create a database for the two communtiies. One for the top communities, one for the mid communtities. 

In [19]:
communtiy_summarizer.find_comunity_summary(global_community_inputs)

Generating summary for community 14
Generating summary for community 15
Generating summary for community 11
Generating summary for community 13
Generating summary for community 9
Generating summary for community 3
Generating summary for community 10
Generating summary for community 5
Generating summary for community 0
Generating summary for community 1
Generating summary for community 6
Generating summary for community 7


In [20]:
global_communities_objects = transform_dicts_to_objects(global_community_inputs)

In [22]:
communtiy_summarizer.find_comunity_summary(mid_community_inputs)

Generating summary for community 14
Generating summary for community 15
Generating summary for community 13
Generating summary for community 10
Generating summary for community 0
Generating summary for community 4
Generating summary for community 5
Generating summary for community 1
Generating summary for community 12
Generating summary for community 7
Generating summary for community 2
Generating summary for community 3
Generating summary for community 9
Generating summary for community 11


In [23]:
mid_community_objects = transform_dicts_to_objects(mid_community_inputs)

# Create global communtiy database

In [24]:
global_retriver = generate_database_and_retriever(
    chroma_index_folder="chroma_index",
    raw_data_folder="raw_data",
    main_folder="./localdb/global_communities",
    db_name="multi_modal_rag",
    ollama_model_name="embeddinggemma:latest",
)

mid_retriver = generate_database_and_retriever(
    chroma_index_folder="chroma_index",
    raw_data_folder="raw_data",
    main_folder="./localdb/mid_communities",
    db_name="multi_modal_rag",
    ollama_model_name="embeddinggemma:latest",
)

In [25]:
populate_community_database(global_retriver, global_communities_objects, "global")

Added 12 global communities to the retriever.


In [26]:
populate_community_database(mid_retriver, mid_community_objects, "mid")

Added 14 mid communities to the retriever.


# Create a DB for the nodes

In [25]:
node_retriver = generate_database_and_retriever(
    chroma_index_folder="chroma_index",
    raw_data_folder="raw_data",
    main_folder="./localdb/node_db",
    db_name="multi_modal_rag",
    ollama_model_name="embeddinggemma:latest",
)

In [22]:
# Get all nodes
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    nodes = fetch_nodes_data(driver)

In [26]:
populate_node_db(node_retriver, nodes)

Added 60 nodes to the retriever.
